# Notebook 1 — Method Overview: White-Box Lie Detection via Representation Engineering

## Project context

This project reproduces the **white-box lie-detection** methodology from the paper:

> **Representation Engineering: A Top-Down Approach to AI Transparency**
> Andy Zou, Long Phan, Sarah Chen, James Campbell, Phillip Guo, Richard Ren, Alexander Pan, Xuwang Yin, Mantas Mazeika, Ann-Kathrin Dombrowski, Shashwat Goel, Nathaniel Li, Michael J. Byun, Zifan Wang, Alex Mallen, Steven Basart, Sanmi Koyejo, Dawn Song, Matt Fredrikson, J. Zico Kolter, Dan Hendrycks — 2023

The methodology is also described in the LessWrong post *"Representation Engineering: Mistral-7B on Honesty"* and implemented in the [repeng](https://github.com/mishajw/repeng) GitHub repository by Misha Wagner.

### What is the goal?

We want to determine whether a language model is "lying" — that is, whether a statement it processes is factually true or false — **without asking the model to generate any text**. Instead, we look directly at its internal representations (hidden states) and train a simple linear classifier (a "probe") to read off truthfulness from those vectors. This is called a **white-box** approach because we open up the model and inspect what is inside, rather than treating it as a black box that only produces text outputs.

### Why does this matter?

As LLMs become more capable, they can produce fluent, confident-sounding text that is factually wrong (hallucinations) or deliberately deceptive (if fine-tuned or prompted adversarially). A white-box lie detector would give us a tool to audit model internals independently of what the model says, providing an additional safety layer that does not rely on the model's own honesty.

## 1. What is Representation Engineering (RepE)?

### The core idea

Transformer language models are built from a stack of layers. As a prompt flows through the model, each layer transforms the input into increasingly abstract **hidden-state vectors**. These vectors live in a high-dimensional space (for GPT-2 Medium, $\mathbb{R}^{1024}$).

**Representation Engineering** (RepE) is the hypothesis — supported by growing empirical evidence — that many human-interpretable concepts are encoded as **linear directions** in this hidden-state space. In other words, there exists a direction vector $\mathbf{d}$ such that projecting any hidden state onto $\mathbf{d}$ tells you how strongly that concept is present. This has been observed for concepts like sentiment, toxicity, and — crucially for this project — **truthfulness**.

This is sometimes called the **linear representation hypothesis**: if you can find the right direction, a simple dot product is enough to read off the concept. No deep neural network is needed on top of the hidden states — just a linear probe.

### The two-stage framework

The RepE paper describes two complementary operations:

| Stage | Name | What it does | Output |
|-------|------|-------------|--------|
| **Stage 1** | **Reading** (probing) | Extract and analyse hidden states to find the linear direction encoding a concept | A trained probe (classifier) that can detect the concept |
| **Stage 2** | **Writing** (steering) | Add or subtract the concept direction from hidden states during inference to change the model's behaviour | A control vector that makes the model more or less truthful, toxic, etc. |

**In this project, we focus entirely on Stage 1 — Reading.** We train probes to detect whether the model's hidden states encode truthfulness, and we study how well these probes generalise across different types of questions. The writing/steering stage is what the [repeng](https://github.com/mishajw/repeng) library implements with its `ControlVector` API, but we do not use it here.

## 2. The truthfulness hypothesis and contrast pairs

### The hypothesis

The central claim we are testing is:

> **Language models internally encode whether a statement is true or false as a linear direction in their hidden-state space.**

If this hypothesis holds, then for any prompt containing a factual claim, the model's hidden states should be systematically different depending on whether the claim is true or false — and this difference should be capturable by a simple linear classifier.

### Building contrast pairs

To find the truthfulness direction, we need training data where the only thing that varies is whether the answer is correct. We achieve this by constructing **contrast pairs**: two prompts that share the same question but differ in the answer.

For example, for a geography question about Paris:

```
Prompt A (TRUE):
  Consider the correctness of the answer to the following question:
  Question: Which country contains the city Paris?
  Answer: France
  The probability of the answer being correct is

Prompt B (FALSE):
  Consider the correctness of the answer to the following question:
  Question: Which country contains the city Paris?
  Answer: Italy
  The probability of the answer being correct is
```

Both prompts have identical structure, identical question, and identical framing. The only difference is the answer: "France" (correct) vs. "Italy" (incorrect). Any systematic difference in hidden states between these two prompts can therefore be attributed to the model's internal representation of truthfulness.

In practice, we often have more than two candidates per question (e.g. 4 candidate countries for a geography question), which gives us richer contrast sets.

### Why the last token?

We extract the hidden state at the **last token position** of the prompt (the token right before the model would generate its next output). This is the most informative position because:

1. **Full context attention**: In a causal (autoregressive) transformer like GPT-2, each token can only attend to tokens that came before it. The last token is the only position that has attended to the entire prompt — the question, the answer, and the framing.

2. **Decision point**: The last token is where the model must "decide" what to generate next. At this point, it has already processed both the question and the candidate answer, and its hidden state reflects its internal assessment of the relationship between them.

3. **No generation needed**: Critically, the model never actually generates any text. We run a single forward pass, collect the hidden states at the last position from every layer, and discard the model's output logits. This makes the approach purely observational — we are reading the model's internal state, not asking it to express an opinion.

For GPT-2 Medium, each prompt produces a tensor of shape $(24, 1024)$: one 1024-dimensional vector for each of the 24 transformer layers. We then select a single layer and work with its 1024-dimensional vector.

## 3. The four probe methods

Once we have extracted hidden-state vectors labelled as "true" or "false", we train a **linear probe** — a classifier that finds a hyperplane separating the two classes. We implement four different methods, each with different assumptions and trade-offs.

### 3.1 Difference in Means (DIM)

The simplest possible approach. We compute the centroid (mean vector) of all true hidden states and the centroid of all false hidden states, and define the truthfulness direction as the normalised difference between them:

$$\mathbf{d}_{\text{DIM}} = \frac{\boldsymbol{\mu}^{+} - \boldsymbol{\mu}^{-}}{\|\boldsymbol{\mu}^{+} - \boldsymbol{\mu}^{-}\|}$$

where $\boldsymbol{\mu}^{+} = \frac{1}{N^+}\sum_{i: y_i=1} \mathbf{h}_i$ is the mean of all true vectors and $\boldsymbol{\mu}^{-}$ is the mean of all false vectors.

To classify a new vector $\mathbf{h}$, we compute its projection onto this direction relative to the midpoint $\mathbf{c} = (\boldsymbol{\mu}^{+} + \boldsymbol{\mu}^{-}) / 2$:

$$\text{score}(\mathbf{h}) = (\mathbf{h} - \mathbf{c}) \cdot \mathbf{d}_{\text{DIM}}$$

A positive score indicates "true", negative indicates "false".

**Pros**: Closed-form solution, no iterative fitting, no hyperparameters, extremely fast.
**Cons**: Ignores within-class variance entirely. If the true and false distributions overlap heavily or have different shapes, DIM may find a suboptimal direction.

### 3.2 Linear Algebraic Treatment (LAT) — the RepE paper's main method

LAT is the signature method from the RepE paper. It finds the truthfulness direction **without explicit supervision on individual examples**. Instead, it exploits the structure of contrast pairs:

1. **Pair activations**: Take hidden states and randomly pair them (without replacement), forming pairs $(\mathbf{h}_i^{(A)}, \mathbf{h}_i^{(B)})$.
2. **Compute differences**: For each pair, compute $\boldsymbol{\delta}_i = \mathbf{h}_i^{(A)} - \mathbf{h}_i^{(B)}$.
3. **Centre**: Subtract the mean: $\tilde{\boldsymbol{\delta}}_i = \boldsymbol{\delta}_i - \bar{\boldsymbol{\delta}}$.
4. **PCA**: Run Principal Component Analysis on the set $\{\tilde{\boldsymbol{\delta}}_i\}$ and take the **first principal component** (the direction of maximum variance).

The intuition is: if true and false vectors consistently differ along a particular direction, then when we subtract random pairs, that direction will show the largest variance in the difference vectors. PCA automatically identifies it.

**Pros**: Does not need per-example labels — only needs prompts designed so that truth and falsehood vary systematically. This makes it more robust in settings where labels are noisy or unavailable.
**Cons**: Assumes the truth direction accounts for the largest variance in the contrast space, which may not hold if other confounds (e.g., topic, answer length) vary more.

### 3.3 Logistic Regression (LR)

The standard supervised baseline. We fit an L2-regularised logistic regression model:

$$P(y = 1 \mid \mathbf{h}) = \sigma(\mathbf{w}^T \mathbf{h} + b)$$

where $\sigma$ is the sigmoid function, $\mathbf{w} \in \mathbb{R}^{1024}$ is the weight vector, and $b$ is the bias. The model is trained by minimising the regularised cross-entropy loss.

The decision boundary is a hyperplane (since logistic regression is a linear model), so this is still a linear probe — but one that is optimised to maximise classification accuracy on the training set.

**Pros**: Generally achieves the highest in-distribution accuracy because it directly optimises for classification. Well-understood, easy to implement with scikit-learn.
**Cons**: Can overfit to dataset-specific features (e.g., the probe might learn that geography questions tend to have certain hidden-state patterns) rather than finding a universal truth direction. This can hurt cross-domain generalisation.

### 3.4 Grouped PCA (PCA-G)

A refinement of LAT that explicitly removes between-group variation before applying PCA:

1. **Within-group centring**: For each question group $g$, compute the group mean $\boldsymbol{\mu}_g$ and subtract it from all members: $\tilde{\mathbf{h}}_i = \mathbf{h}_i - \boldsymbol{\mu}_g$ for all $i \in g$.
2. **PCA**: Stack all centred vectors and run PCA. Take the first principal component.

The within-group centring removes all variation that is due to the *topic* of the question (e.g., geography vs. arithmetic) and forces PCA to focus exclusively on the *within-group* contrast — which is precisely the true-vs-false distinction. This should produce a purer truthfulness direction than vanilla LAT.

**Pros**: By removing topic-level variation, PCA-G isolates the truth signal more cleanly. Often generalises better across domains.
**Cons**: Requires group structure in the data (which we always have in our setup). Slightly more complex to implement.

## 4. Our experimental setup

### Model: Microsoft Phi-2 (2.7B parameters, 32 layers)

We use **Phi-2** (`microsoft/phi-2`), a 2.7-billion-parameter transformer with:
- **32 transformer layers** (vs. 6 for distilgpt2, 24 for gpt2-medium)
- **2560-dimensional hidden states** (vs. 768 for distilgpt2, 1024 for gpt2-medium)
- Trained on high-quality textbook-style data, giving it strong factual knowledge

The original RepE paper uses **Llama-2-13B-chat** (13B parameters, 40 layers). Phi-2 is smaller but still large enough to develop rich internal representations of truthfulness. It can run on a standard laptop with 16GB RAM using float16 precision.

### Datasets: 8 datasets, 1903 prompts, 544 question groups

We use a mix of hand-crafted datasets and standard benchmarks from the RepE paper:

| Dataset | Source | Groups | Prompts | Type |
|---------|--------|--------|---------|------|
| **cities** | Hand-crafted | 30 | 120 | Geography (4 candidates) |
| **larger_than** | Hand-crafted | 30 | 60 | Numerical reasoning (2 candidates) |
| **qa** | Hand-crafted | 30 | 120 | General knowledge (4 candidates) |
| **repeng_truthful** | [repeng repo](https://github.com/mishajw/repeng) | 54 | 108 | Self-report honesty (2 candidates) |
| **truthfulqa** | [TruthfulQA benchmark](https://huggingface.co/datasets/truthful_qa) | 100 | ~494 | Common misconceptions (variable) |
| **arc_easy** | [ARC benchmark](https://huggingface.co/datasets/allenai/ai2_arc) | 100 | ~402 | Easy science exams (3-5 candidates) |
| **arc_challenge** | [ARC benchmark](https://huggingface.co/datasets/allenai/ai2_arc) | 100 | ~399 | Hard science exams (3-5 candidates) |
| **boolq** | [BoolQ benchmark](https://huggingface.co/datasets/google/boolq) | 100 | 200 | Yes/No questions (2 candidates) |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

import warnings
warnings.filterwarnings("ignore")

from lie_detector_llm.datasets import build_dataset_collection, REPE_TEMPLATE

collection = build_dataset_collection()
df = collection.frame

# Dataset statistics
stats = df.groupby("dataset_name").agg(
    n_rows=("dataset_name", "size"),
    n_groups=("group_id", "nunique"),
    n_true=("label", "sum"),
).reset_index()
stats["n_false"] = stats["n_rows"] - stats["n_true"]
print(f"Total: {len(df)} prompts, {df['group_id'].nunique()} groups across {len(stats)} datasets\n")
print(stats.to_string(index=False))

print("\n\n--- Prompt template ---\n")
print(REPE_TEMPLATE)

print("\n\n--- Example group (Paris) ---\n")
group = df[df["group_id"] == "cities::Paris"].sort_values("label", ascending=False)
for _, row in group.iterrows():
    marker = "TRUE " if row["label"] else "FALSE"
    print(f"  [{marker}]  Answer: {row['answer']}")